# IR-Copilot — Step 1: Data Extraction & Financial Fact Store

**Goal of this step:** turn a ticker + quarter into a typed **Financial Fact Store** where every
number carries provenance (source + URL + as-of date). This is the grounding backbone that makes
"no missing / no hallucinated numbers" possible — see `docs/grounding.md`.

What this notebook does:
1. Load all config from `.env` (no hard-coded settings).
2. Fetch financial metrics from **defeatbeta-api** (with a deterministic **mock** fallback so the
   demo runs offline) and a **yfinance** fallback.
3. Build a typed `FinancialFact` store with stable `fact_id`s and record any gaps.
4. Persist the store to git-ignored `artifacts/` (parquet + json) and reload it.
5. Demonstrate the **numeric grounding guard** + **coverage check** that later agents rely on.

> Dependencies come from `requirements.txt` (no `!pip install` here).

In [1]:
from __future__ import annotations
import os, json, re, datetime as dt
from pathlib import Path
from typing import Optional
from dotenv import load_dotenv

# Anchor to the project root (where .env / requirements.txt live) so artifacts land in one
# place regardless of the notebook's working directory.
def project_root() -> Path:
    here = Path.cwd()
    for d in (here, *here.parents):
        if (d / "requirements.txt").exists() or (d / ".env").exists():
            return d
    return here

ROOT = project_root()
load_dotenv(ROOT / ".env")  # read .env (git-ignored)

TICKER        = os.getenv("DEMO_TICKER", "NVDA")
PERIOD        = os.getenv("DEMO_PERIOD", "FY2026Q1")
USE_MOCK      = os.getenv("USE_MOCK_DATA", "true").lower() == "true"
ARTIFACTS_DIR = (ROOT / os.getenv("ARTIFACTS_DIR", "artifacts")).resolve()
FACT_DIR      = ARTIFACTS_DIR / "fact_store"
FACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker={TICKER}  Period={PERIOD}  USE_MOCK_DATA={USE_MOCK}")
print(f"Artifacts -> {FACT_DIR.resolve()}")

Ticker=NVDA  Period=FY2026Q1  USE_MOCK_DATA=True
Artifacts -> /Users/Shared/0projects/amdhackathonfin/artifacts/fact_store


## 1. The Financial Fact Store

Numbers live **only** here. Downstream the LLM receives `fact_id`s and writes *slots*
(e.g. `{{F-0003}}`) — it never emits free-form digits. Each fact records where it came from.

In [2]:
import pandas as pd
from pydantic import BaseModel

class FinancialFact(BaseModel):
    fact_id: str
    ticker: str
    metric: str
    period: str
    value: float
    unit: str               # "USD" | "%" | "x"
    source: str             # "defeatbeta-api:ttm_eps" | "yfinance:..." | "mock:..."
    source_url: Optional[str] = None
    as_of: dt.date

class FactStore:
    def __init__(self, ticker: str, period: str):
        self.ticker, self.period = ticker, period
        self._facts: list[FinancialFact] = []
        self.gaps: list[dict] = []
        self._n = 0

    def add(self, metric, value, unit, source, source_url=None, as_of=None) -> FinancialFact:
        self._n += 1
        fact = FinancialFact(
            fact_id=f"F-{self._n:04d}", ticker=self.ticker, metric=metric,
            period=self.period, value=float(value), unit=unit, source=source,
            source_url=source_url, as_of=as_of or dt.date.today())
        self._facts.append(fact)
        return fact

    def record_gap(self, metric, reason):
        self.gaps.append({"metric": metric, "reason": str(reason)[:160]})

    @property
    def facts(self) -> list[FinancialFact]:
        return self._facts

    def get(self, metric) -> Optional[FinancialFact]:
        return next((f for f in self._facts if f.metric == metric), None)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([f.model_dump() for f in self._facts])

## 2. Extraction specs

Each ratio maps to a single `defeatbeta-api` `Ticker` method (confirmed from the project docs,
see `docs/data-sources.md`). `MOCK` holds a realistic snapshot used when `USE_MOCK_DATA=true`
or when a live call fails — so the pipeline always completes.

In [3]:
# (metric, defeatbeta-api Ticker method, unit)
SPECS = [
    ("ttm_eps",            "ttm_eps",                      "USD"),
    ("ttm_pe",             "ttm_pe",                       "x"),
    ("market_cap",         "historical_market_cap",        "USD"),
    ("ps_ratio",           "historical_ps_ratio",          "x"),
    ("pb_ratio",           "historical_pb_ratio",          "x"),
    ("peg_ratio",          "historical_peg_ratio",         "x"),
    ("roe",                "historical_roe",               "%"),
    ("roa",                "historical_roa",               "%"),
    ("roic",               "historical_roic",              "%"),
    ("wacc",               "historical_wacc",              "%"),
    ("equity_multiplier",  "historical_equity_multiplier", "x"),
    ("asset_turnover",     "historical_asset_turnover",    "x"),
]

# Deterministic demo snapshot (illustrative figures for the offline/mock path).
MOCK = {
    "ttm_eps": 3.10, "ttm_pe": 38.5, "market_cap": 3.02e12, "ps_ratio": 28.4,
    "pb_ratio": 45.1, "peg_ratio": 1.2, "roe": 91.3, "roa": 55.2, "roic": 78.0,
    "wacc": 11.5, "equity_multiplier": 1.65, "asset_turnover": 0.95,
    "revenue": 44.06e9, "gross_margin": 75.0, "operating_margin": 64.9,
}
MOCK_URL = "https://huggingface.co/datasets/defeat-beta/yahoo-finance-data"
UNITS = {m: u for m, _, u in SPECS} | {
    "revenue": "USD", "gross_margin": "%", "operating_margin": "%"}

## 3. Build the store

Live path: `defeatbeta-api`. If unavailable or a metric fails, we fall back to the mock value and
**still record the gap** (nothing is silently dropped).

In [4]:
def latest_numeric(obj) -> float:
    # Best-effort: pull the most recent numeric value from a defeatbeta-api result.
    df = getattr(obj, "data", obj)
    if isinstance(df, pd.DataFrame) and not df.empty:
        num = df.select_dtypes("number")
        if num.shape[1]:
            return float(num.iloc[-1, -1])
    if isinstance(obj, (int, float)):
        return float(obj)
    raise ValueError(f"could not parse numeric from {type(obj).__name__}")

store = FactStore(TICKER, PERIOD)

ticker_obj = None
if not USE_MOCK:
    try:
        from defeatbeta_api.data.ticker import Ticker
        ticker_obj = Ticker(TICKER)
    except Exception as e:
        print(f"defeatbeta-api unavailable ({e}); using mock snapshot for all metrics.")

for metric, method, unit in SPECS:
    if ticker_obj is not None:
        try:
            val = latest_numeric(getattr(ticker_obj, method)())
            store.add(metric, val, unit, source=f"defeatbeta-api:{method}")
            continue
        except Exception as e:
            store.record_gap(metric, f"live fetch failed: {e}")
    if metric in MOCK:                       # mock / fallback
        store.add(metric, MOCK[metric], unit, source="mock:defeatbeta-snapshot",
                  source_url=MOCK_URL)
    else:
        store.record_gap(metric, "no live value and no mock")

# Required income-statement metrics (sourced from quarterly_income_statement live;
# from the mock snapshot here).
for metric in ["revenue", "gross_margin", "operating_margin"]:
    if store.get(metric) is None and metric in MOCK:
        store.add(metric, MOCK[metric], UNITS[metric],
                  source="mock:defeatbeta-snapshot", source_url=MOCK_URL)

print(f"Extracted {len(store.facts)} facts; {len(store.gaps)} gap(s): {store.gaps}")
store.to_dataframe()[["fact_id", "metric", "value", "unit", "source"]]

Extracted 15 facts; 0 gap(s): []


,fact_id,metric,value,unit,source
0,F-0001,ttm_eps,3.100000e+00,USD,mock:defeatbeta-snapshot
1,F-0002,ttm_pe,3.850000e+01,x,mock:defeatbeta-snapshot
2,F-0003,market_cap,3.020000e+12,USD,mock:defeatbeta-snapshot
3,F-0004,ps_ratio,2.840000e+01,x,mock:defeatbeta-snapshot
4,F-0005,pb_ratio,4.510000e+01,x,mock:defeatbeta-snapshot
5,F-0006,peg_ratio,1.200000e+00,x,mock:defeatbeta-snapshot
6,F-0007,roe,9.130000e+01,%,mock:defeatbeta-snapshot
7,F-0008,roa,5.520000e+01,%,mock:defeatbeta-snapshot
8,F-0009,roic,7.800000e+01,%,mock:defeatbeta-snapshot
9,F-0010,wacc,1.150000e+01,%,mock:defeatbeta-snapshot


## 4. yfinance fallback (demonstration)

If a metric is still missing from the primary source, fall back to `yfinance` and tag provenance
honestly so the source is always traceable.

In [5]:
def yfinance_market_cap(ticker: str) -> Optional[float]:
    try:
        import yfinance as yf
        fi = yf.Ticker(ticker).fast_info
        return float(getattr(fi, "market_cap", None) or fi["market_cap"])
    except Exception as e:
        print(f"yfinance fallback unavailable: {e}")
        return None

if store.get("market_cap") is None:
    mc = yfinance_market_cap(TICKER)
    if mc:
        store.add("market_cap", mc, "USD", source="yfinance:fast_info.market_cap",
                  source_url=f"https://finance.yahoo.com/quote/{TICKER}")

mc = store.get("market_cap")
print("market_cap:", f"{mc.value:,.0f} {mc.unit} (via {mc.source})" if mc else "MISSING")

market_cap: 3,020,000,000,000 USD (via mock:defeatbeta-snapshot)


## 5. Persist & reload

Artifacts are written to the **git-ignored** `artifacts/` directory (parquet for analytics, json
for inspection), then reloaded to prove round-tripping.

In [6]:
df = store.to_dataframe()
parquet_path = FACT_DIR / f"{TICKER}_{PERIOD}.parquet"
json_path    = FACT_DIR / f"{TICKER}_{PERIOD}.json"

df.to_parquet(parquet_path, index=False)
json_path.write_text(json.dumps([json.loads(f.model_dump_json()) for f in store.facts], indent=2))
print("saved:", parquet_path)
print("saved:", json_path)

reloaded = pd.read_parquet(parquet_path)
print(f"reloaded {len(reloaded)} facts")
reloaded[["fact_id", "metric", "period", "value", "unit", "source", "as_of"]]

saved: /Users/Shared/0projects/amdhackathonfin/artifacts/fact_store/NVDA_FY2026Q1.parquet
saved: /Users/Shared/0projects/amdhackathonfin/artifacts/fact_store/NVDA_FY2026Q1.json
reloaded 15 facts


,fact_id,metric,period,value,unit,source,as_of
0,F-0001,ttm_eps,FY2026Q1,3.100000e+00,USD,mock:defeatbeta-snapshot,2026-06-10
1,F-0002,ttm_pe,FY2026Q1,3.850000e+01,x,mock:defeatbeta-snapshot,2026-06-10
2,F-0003,market_cap,FY2026Q1,3.020000e+12,USD,mock:defeatbeta-snapshot,2026-06-10
3,F-0004,ps_ratio,FY2026Q1,2.840000e+01,x,mock:defeatbeta-snapshot,2026-06-10
4,F-0005,pb_ratio,FY2026Q1,4.510000e+01,x,mock:defeatbeta-snapshot,2026-06-10
5,F-0006,peg_ratio,FY2026Q1,1.200000e+00,x,mock:defeatbeta-snapshot,2026-06-10
6,F-0007,roe,FY2026Q1,9.130000e+01,%,mock:defeatbeta-snapshot,2026-06-10
7,F-0008,roa,FY2026Q1,5.520000e+01,%,mock:defeatbeta-snapshot,2026-06-10
8,F-0009,roic,FY2026Q1,7.800000e+01,%,mock:defeatbeta-snapshot,2026-06-10
9,F-0010,wacc,FY2026Q1,1.150000e+01,%,mock:defeatbeta-snapshot,2026-06-10


## 6. Grounding guard + coverage check

Two checks every later agent relies on (`docs/grounding.md`):
- **Numeric grounding** — every number in generated text must match a Fact-Store value
  (allowing `$bn`/`$tn` and percent forms). Unmatched numbers are flagged.
- **Coverage** — every *required* metric for the period must be present.

In [7]:
NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")

def extract_numbers(text: str) -> list[float]:
    out = []
    for tok in NUM_RE.findall(text):
        try:
            out.append(float(tok.replace(",", "")))
        except ValueError:
            pass
    return out

def allowed_values(store: FactStore) -> set[float]:
    vals: set[float] = set()
    for f in store.facts:
        vals.add(round(f.value, 4))
        if f.unit == "USD" and abs(f.value) >= 1e9:
            vals.add(round(f.value / 1e9, 4))    # "$44.06 billion"
            vals.add(round(f.value / 1e12, 4))   # "$3.02 trillion"
        if f.unit == "%":
            vals.add(round(f.value / 100, 4))    # 0.75 form
    return vals

def check_numbers(text: str, store: FactStore, tol: float = 0.02) -> list[float]:
    allowed = allowed_values(store)
    return [n for n in extract_numbers(text)
            if not any(abs(n - a) <= tol * max(1.0, abs(a)) for a in allowed)]

eps = store.get("ttm_eps").value
grounded     = f"TTM EPS was {eps} and gross margin was 75.0%."
hallucinated = "TTM EPS was 9.99 and gross margin was 88.8%."

print("grounded     -> ungrounded numbers:", check_numbers(grounded, store))
print("hallucinated -> ungrounded numbers:", check_numbers(hallucinated, store))

assert check_numbers(grounded, store) == [], "grounded sentence should pass"
assert check_numbers(hallucinated, store),    "hallucinated sentence must be flagged"

REQUIRED = {"revenue", "ttm_eps", "gross_margin", "operating_margin", "market_cap"}
present  = {f.metric for f in store.facts}
missing  = REQUIRED - present
print("coverage missing:", missing or "none")
assert not missing, f"missing required metrics: {missing}"

print()
print("Grounding guard works: hallucinated numbers flagged, coverage complete.")

grounded     -> ungrounded numbers: []
hallucinated -> ungrounded numbers: [9.99, 88.8]
coverage missing: none

Grounding guard works: hallucinated numbers flagged, coverage complete.


## Done — Step 1

We have a typed, persisted **Financial Fact Store** with provenance, plus the numeric-grounding
and coverage guards the verifier will enforce.

**Next (Step 2):** the multimodal **Knowledge Wiki** — chunk + embed transcripts/filings into
Qdrant with reranked, cited retrieval (`docs/wiki-ingestion.md`).